# Session 5: a third clip
**A. Real-world problem.** Two clips at 30 and 24 fps cannot tell 22.5 rev/s from 142.5, 262.5 or
382.5 rev/s: the answer repeats every lcm(30, 24) = 120 rev/s. A fast machine, a turbine or a drill, can easily turn
faster than that. A third clip at 25 fps stretches the unambiguous range to 600 rev/s. Why, which third frame
rate helps and which one does not, and what happens to the noise?

**Plan.** (1) the speeds that sit on all three ladders, (2) why the range is the lcm of all rates and
when a third rate does not help, (3) choosing the third rate, (4) recovery from noisy clips with
three numbers, and what the extra clip does to the uncertainty.

In [ ]:
import math
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"


def signed_alias(f, fs):
    """Apparent speed of a wheel turning f rev/s filmed at fs fps, in (-fs/2, fs/2] (negative = looks backwards)."""
    return f - fs * math.floor(f / fs + 0.5)


def candidates(a, fs, f_lo, f_hi):
    """Speeds in [f_lo, f_hi] that look like a at frame rate fs: the rungs a + k fs."""
    k0 = math.ceil((f_lo - a) / fs - 1e-9)
    k1 = math.floor((f_hi - a) / fs + 1e-9)
    return [a + k * fs for k in range(k0, k1 + 1)]

## Step 1: the speeds that sit on every ladder
Each clip allows a ladder of rungs a + k fs. A speed that is on all the ladders at once is compatible
with all the clips. With two clips there are several of them; with a third, almost all disappear.

In [ ]:
def common_speeds(apparents, rates, f_lo=0.0, f_hi=400.0, tol=1e-6):
    """Speeds in [f_lo, f_hi] on every ladder (within tol). One entry means the answer is unique."""
    if len(apparents) != len(rates) or len(rates) < 2:
        raise ValueError("give one apparent speed per frame rate, and at least two clips")
    ladders = [candidates(a, fs, f_lo, f_hi) for a, fs in zip(apparents, rates)]
    out = []
    for x in ladders[0]:
        if all(any(abs(x - y) <= tol for y in lad) for lad in ladders[1:]):
            out.append(round(x, 6))
    return out

## Step 2: the range is the lcm of all the rates
A wrong speed f' = f + d looks the same as f on a clip at fs when d is a whole number of frame
rates, d = k fs. It looks the same on all the clips only when d is a common multiple of all the rates: d = k lcm(fs1, fs2, fs3). So the
unambiguous range is the least common multiple of *all* the rates. A third rate adds
nothing when it divides the lcm of the other two (20, 24, 30, 40 or 60 fps next to 30 and 24, whose lcm is 120).

In [ ]:
def unambiguous_range(rates):
    """With whole-number rates and signed measurements, f and f + this look the same on every clip."""
    if any(int(round(r)) != r or r <= 0 for r in rates):
        raise ValueError("this rule needs positive whole-number frame rates")
    return int(np.lcm.reduce([int(r) for r in rates]))


def third_rate_helps(rate_a, rate_b, rate_c):
    """True when a third clip at rate_c extends the range of the pair (rate_a, rate_b)."""
    return unambiguous_range([rate_a, rate_b, rate_c]) > unambiguous_range([rate_a, rate_b])


def rate_table(rate_a, rate_b, third_rates):
    """(third rate, range of the three rates, does it help) for each candidate third rate."""
    return [(r, unambiguous_range([rate_a, rate_b, r]), third_rate_helps(rate_a, rate_b, r)) for r in third_rates]

## Step 3: noisy clips and three numbers
With noise (Session 4), pick the set of rungs, one per ladder, whose members are closest to each other:
the score of a set is the sum of squared distances of the rungs from their mean, and the
answer is the mean. For equal noise on every clip this is the least-squares answer. Its error is the mean of n
independent errors, so the standard deviation is sigma / sqrt(n): three clips give 0.17 rev/s where two give 0.21 rev/s for sigma = 0.3, and the chance of being
within t is erf(t sqrt(n) / (sigma sqrt 2)). As in Session 4, the rungs are searched 4 sigma beyond the range
and the answer is clipped back into it.

In [ ]:
MAX_SETS = 400_000


def rung_sets(apparents, rates, f_lo, f_hi, margin=0.0):
    """All sets of one rung per ladder: (rungs array [n_sets, n_clips])."""
    ladders = [np.array(candidates(a, fs, f_lo - margin, f_hi + margin)) for a, fs in zip(apparents, rates)]
    if any(len(lad) == 0 for lad in ladders):
        raise ValueError("a ladder has no rung inside the range")
    if np.prod([len(lad) for lad in ladders], dtype=float) > MAX_SETS:
        raise ValueError("too many rung sets: narrow the range")
    grid = np.meshgrid(*ladders, indexing="ij")
    return np.stack([g.ravel() for g in grid], axis=1)


def recover_noisy_n(apparents, rates, sigma, f_lo=5.0, f_hi=399.0):
    """Most likely true speed from n >= 2 noisy apparent speeds (equal noise sigma), with an uncertainty.
    Returns a dict: f (rev/s), sd (rev/s), margin (how many sigma the best set beats the runner-up by)."""
    if sigma < 0:
        raise ValueError("sigma must not be negative")
    if f_hi <= f_lo or len(apparents) != len(rates) or len(rates) < 2 or min(rates) <= 0:
        raise ValueError("need f_hi > f_lo, positive rates and one apparent speed per rate (at least two)")
    sets = rung_sets(apparents, rates, f_lo, f_hi, 4.0 * sigma)
    mean = sets.mean(axis=1)
    spread = ((sets - mean[:, None]) ** 2).sum(axis=1)
    order = np.argsort(spread)
    runner = spread[order[1]] if len(order) > 1 else math.inf
    margin = (math.sqrt(runner) - math.sqrt(spread[order[0]])) / sigma if sigma > 0 else math.inf
    return {"f": float(min(max(mean[order[0]], f_lo), f_hi)), "sd": sigma / math.sqrt(len(rates)), "margin": float(margin)}


def predicted_within(sigma, n_clips, tol=0.5):
    """Chance that the mean of the true rungs is within tol of the truth: erf(tol sqrt(n) / (sigma sqrt 2))."""
    return 1.0 if sigma <= 0 else math.erf(tol * math.sqrt(n_clips) / (sigma * math.sqrt(2.0)))


def run_trials(rates, sigma, n_trials=1000, seed=0, f_lo=5.0, f_hi=399.0, tol=0.5):
    """Share of trials (true speed uniform in the range) in which the recovery is within tol of the truth."""
    rng = np.random.default_rng(seed)
    good = 0
    for _ in range(n_trials):
        f = rng.uniform(f_lo, f_hi)
        a = [signed_alias(f, r) + rng.normal(0.0, sigma) for r in rates]
        good += abs(recover_noisy_n(a, rates, sigma, f_lo, f_hi)["f"] - f) <= tol
    return good / n_trials

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    f_true = 22.5
    a30, a24, a25 = (signed_alias(f_true, r) for r in (30, 24, 25))
    print(f"apparent speeds of {f_true} rev/s: {a30} (30 fps), {a24} (24 fps), {a25} (25 fps)")
    print("two clips (30, 24), range 0 to 400:", common_speeds([a30, a24], [30, 24]))
    print("three clips (30, 24, 25), range 0 to 400:", common_speeds([a30, a24, a25], [30, 24, 25]))
    print("three clips, range 0 to 1000:", common_speeds([a30, a24, a25], [30, 24, 25], 0, 1000), "(repeats at 22.5 + 600)")
    print("\nthird rate next to 30 and 24 fps: range of the three rates")
    for r, rng_, helps in rate_table(30, 24, [20, 24, 25, 26, 27, 29, 36, 40, 45, 50]):
        print(f"  {r} fps: range {rng_:5d} rev/s, {'helps' if helps else 'no help'}")
    print("\nshare within 0.5 rev/s, true speed uniform 5 to 399 rev/s, 600 trials:")
    sig_list = [0.1, 0.2, 0.3, 0.4, 0.6, 0.8, 1.0]
    two = [run_trials([30, 24], s, 600) for s in sig_list]
    three = [run_trials([30, 24, 25], s, 600) for s in sig_list]
    print(f"{'noise':>6s} {'two clips':>10s} {'three clips':>12s} {'limit (3)':>10s}")
    for s, a, b in zip(sig_list, two, three):
        print(f"{s:6.1f} {a:10.3f} {b:12.3f} {predicted_within(s, 3):10.3f}")

    fig, axs = plt.subplots(2, 2, figsize=(11.5, 8))
    ax = axs[0, 0]
    rows = [(30, a30, "tab:blue"), (24, a24, "tab:orange"), (25, a25, "tab:green")]
    for i, (r, a, c) in enumerate(rows):
        xs = candidates(a, r, 0, 640)
        ax.plot(xs, np.full(len(xs), 2 - i), "|", ms=16, mew=2.5, color=c)
    ax.axvline(f_true, color="k", ls=":")
    ax.axvline(f_true + 600, color="k", ls=":")
    ax.set_yticks([2, 1, 0])
    ax.set_yticklabels(["30 fps", "24 fps", "25 fps"])
    ax.set_xlim(0, 640)
    ax.set_ylim(-0.6, 2.6)
    ax.set_xlabel("candidate true speed (rev/s)")
    ax.set_title("Three ladders: 22.5 and 622.5 are the only rungs they share below 640", fontsize=9)
    ax = axs[0, 1]
    thirds = list(range(20, 61))
    tab = rate_table(30, 24, thirds)
    ax.bar(thirds, [t[1] for t in tab], color=["tab:green" if t[2] else "tab:red" for t in tab])
    ax.axhline(400, color="k", ls="--", lw=1)
    ax.set_yscale("log")
    ax.set_xlabel("third frame rate (fps), the first two are 30 and 24")
    ax.set_ylabel("unambiguous range (rev/s)")
    ax.set_title("Red: the third rate divides 120 and adds nothing (dashed: 400 rev/s)", fontsize=9)
    ax = axs[1, 0]
    ax.plot(sig_list, two, "o-", color="tab:red", label="two clips (30, 24)")
    ax.plot(sig_list, three, "s-", color="tab:blue", label="three clips (30, 24, 25)")
    ax.plot(sig_list, [predicted_within(s, 3) for s in sig_list], "k:", label="limit for three clips")
    ax.axhline(0.95, color="gray", ls="--")
    ax.set_xlabel("noise on each apparent speed (rev/s)")
    ax.set_ylabel("share within 0.5 rev/s")
    ax.set_title("True speed uniform from 5 to 399 rev/s, 600 trials per point", fontsize=9)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    ax = axs[1, 1]
    rng2 = np.random.default_rng(2)
    e2, e3 = [], []
    for _ in range(1500):
        f = rng2.uniform(5, 59)
        a = [signed_alias(f, r) + rng2.normal(0, 0.5) for r in (30, 24, 25)]
        e3.append(recover_noisy_n(a, [30, 24, 25], 0.5, 5, 59)["f"] - f)
        e2.append(recover_noisy_n(a[:2], [30, 24], 0.5, 5, 59)["f"] - f)
    bins = np.linspace(-1.6, 1.6, 49)
    ax.hist(e2, bins=bins, alpha=0.6, color="tab:red", label=f"two clips, sd {np.std(e2):.3f}")
    ax.hist(e3, bins=bins, alpha=0.6, color="tab:blue", label=f"three clips, sd {np.std(e3):.3f}")
    ax.set_xlabel("recovered minus true speed (rev/s), range 5 to 59, sigma = 0.5")
    ax.set_ylabel("trials (of 1500)")
    ax.set_title("A third clip also lowers the noise: sd falls by sqrt(2/3)", fontsize=9)
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(OUT / "third_clip.png", dpi=110)
    print("saved figure in", OUT)

## Where this stops
Whole-number frame rates, signed apparent speeds measured with equal Gaussian noise, one steady
speed. Real cameras drift in frame rate, and a rate that is only nominally 24 or 25 makes the
rungs slide apart at large k, which limits how far the lcm range is usable. The more clips, the more frames
you must film and the more accurate the rates must be known: the range grows with the rates you add, but each
clip costs time. A fan with several identical blades, a speed that changes and a rolling shutter are the later milestones of the
roadmap.

## Coding tasks
1. **Easy.** For a fan at 31.5 rev/s filmed at 30, 24 and 25 fps, compute the three apparent speeds with `signed_alias`, then print `common_speeds` for the first two clips and for all three, in the range 0 to 400. How many candidates does each give?
2. **Stretch.** Use `rate_table(30, 24, range(20, 61))` to find the third rates whose range is above 400 rev/s, and then the one whose range is the smallest above 400. Then check with `run_trials` (sigma = 0.3) that its share within 0.5 rev/s is not worse than for 29 fps.
3. **Explain.** Explain why the unambiguous range of three clips is the lcm of all three rates and not the largest of the pairwise ranges, using the example 30, 24 and 36 fps.

## Assessment
- **A. Conceptual.** With 30 and 24 fps, a speed 120 rev/s above the true one fools both clips. Why does it no longer fool a third clip at 25 fps?
- **B. Numerical.** The first two rates are 30 and 24 fps. For third rates of 20, 25, 36 and 40 fps give the range of the three rates, and say which make a fan of up to 400 rev/s recoverable.
- **C. Graph and simulation interpretation.** In the top right panel the bars at 36, 45 and 48 fps are green (they help) but lie below the dashed 400 rev/s line. What does that mean for a fan of 380 rev/s filmed with 30, 24 and 45 fps?
- **D. Design and optimisation challenge.** Design: you have clips at 30 and 24 fps and must measure fans up to 400 rev/s with noise 0.3 rev/s. Which third frame rates in 20 to 60 fps work, which one would you choose, and what share of trials will be within 0.5 rev/s?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import math

import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    f = 22.5
    a30, a24, a25 = (L.signed_alias(f, r) for r in (30, 24, 25))

    # ---- the worked example and the roadmap line -----------------------------------------------
    checks.append(("hand calculation: 22.5 rev/s looks like -7.5, -1.5 and -2.5 rev/s at 30, 24 and 25 fps",
                   (a30, a24, a25) == (-7.5, -1.5, -2.5)))
    two = L.common_speeds([a30, a24], [30, 24], 0, 400)
    checks.append(("two clips (30, 24) leave 4 candidates below 400 rev/s: 22.5, 142.5, 262.5 and 382.5", two == [22.5, 142.5, 262.5, 382.5]))
    three = L.common_speeds([a30, a24, a25], [30, 24, 25], 0, 400)
    checks.append(("roadmap line: with 30, 24 and 25 fps the true speed 22.5 rev/s is the only candidate below 400 rev/s", three == [22.5]))
    checks.append(("the range is lcm(30, 24, 25) = 600: 22.5 and 622.5 rev/s look identical on all three clips, and nothing else below 1000 does",
                   L.unambiguous_range([30, 24, 25]) == 600 and L.common_speeds([a30, a24, a25], [30, 24, 25], 0, 1000) == [22.5, 622.5]))

    # ---- independent brute force over the whole range -------------------------------------------
    bad = 0
    for g in np.arange(0.5, 599.6, 0.5):
        aa = [L.signed_alias(float(g), r) for r in (30, 24, 25)]
        bad += L.common_speeds(aa, [30, 24, 25], 0, 599.9) != [float(g)]
    checks.append((f"exact recovery is unique for every speed from 0.5 to 599.5 rev/s in steps of 0.5 ({bad} failures out of 1199)", bad == 0))
    rng = np.random.default_rng(1)
    same = 0
    for _ in range(300):
        g = float(rng.uniform(1, 399))
        aa = [L.signed_alias(g, r) for r in (30, 24, 25)]
        same += len(L.common_speeds(aa, [30, 24, 25], 0, 400, 1e-6)) == 1
    checks.append(("300 random speeds below 400 rev/s: exactly one candidate each time", same == 300))

    # ---- when a third rate helps -----------------------------------------------------------------
    helpless = [r for r in range(20, 61) if not L.third_rate_helps(30, 24, r)]
    brute = [r for r in range(20, 61) if 120 % r == 0]
    checks.append((f"a third rate next to 30 and 24 fps adds nothing exactly when it divides 120: {helpless}", helpless == brute == [20, 24, 30, 40, 60]))
    aa = [L.signed_alias(22.5, r) for r in (30, 24, 20)]
    checks.append(("a third clip at 20 fps changes nothing: still 4 candidates below 400 rev/s (as with two clips)",
                   L.common_speeds(aa, [30, 24, 20], 0, 400) == [22.5, 142.5, 262.5, 382.5]))
    checks.append(("a helpful third rate is not always enough: 36 fps gives only 360 rev/s, so 22.5 and 382.5 both survive below 400",
                   L.unambiguous_range([30, 24, 36]) == 360
                   and L.common_speeds([L.signed_alias(22.5, r) for r in (30, 24, 36)], [30, 24, 36], 0, 400) == [22.5, 382.5]))
    tab = {r: (rg, h) for r, rg, h in L.rate_table(30, 24, [25, 29, 40])}
    checks.append(("range table: third rates 25, 29 and 40 fps give 600, 3480 and 120 rev/s", tab[25] == (600, True) and tab[29] == (3480, True) and tab[40] == (120, False)))

    # ---- noisy clips ------------------------------------------------------------------------------
    t3 = L.run_trials([30, 24, 25], 0.3, 1000, seed=0)
    t2 = L.run_trials([30, 24], 0.3, 1000, seed=0)
    checks.append((f"noisy clips (0.3 rev/s), true speed uniform 5 to 399 rev/s: three clips are within 0.5 rev/s in {t3:.1%} of 1000 trials; two clips in only {t2:.0%} (the range is 120)",
                   t3 >= 0.98 and t2 < 0.40))
    rng = np.random.default_rng(5)
    err, cover = [], 0
    for _ in range(1500):
        g = rng.uniform(5, 59)
        a = [L.signed_alias(g, r) + rng.normal(0, 0.5) for r in (30, 24, 25)]
        r_ = L.recover_noisy_n(a, [30, 24, 25], 0.5, 5, 59)
        err.append(r_["f"] - g)
        cover += abs(r_["f"] - g) <= 2 * r_["sd"]
    err = np.array(err)
    checks.append((f"uncertainty with three clips: sd = sigma / sqrt(3) = {0.5 / math.sqrt(3):.3f}; the errors have sd {err.std():.3f} and the +-2 sd band holds {cover / 1500:.1%} (95.4% expected)",
                   abs(err.std() / (0.5 / math.sqrt(3)) - 1) < 0.07 and abs(cover / 1500 - 0.954) < 0.02))
    ok2 = np.std([L.recover_noisy_n([L.signed_alias(g, r) + rng.normal(0, 0.5) for r in (30, 24)], [30, 24], 0.5, 5, 59)["f"] - g
                  for g in rng.uniform(5, 59, 1500)])
    checks.append((f"a third clip lowers the noise by sqrt(2/3) = 0.82: sd {ok2:.3f} with two clips, {err.std():.3f} with three (sigma = 0.5)", abs(err.std() / ok2 - math.sqrt(2 / 3)) < 0.06))
    pred = all(abs(L.run_trials([30, 24, 25], s, 800, seed=2) - L.predicted_within(s, 3)) < 0.03 for s in (0.4, 0.6))
    checks.append(("the share within 0.5 rev/s follows erf(0.5 sqrt(3) / (sigma sqrt 2)) at sigma = 0.4 and 0.6 (to 3 points): no rule beats it", pred))
    checks.append(("with two clips the same formula gives Session 4's erf(0.5 / sigma)", abs(L.predicted_within(0.3, 2) - math.erf(0.5 / 0.3)) < 1e-12))
    r0 = L.recover_noisy_n([a30, a24, a25], [30, 24, 25], 0.0, 5, 399)
    checks.append(("without noise the answer is exact: 22.5 with sd 0 and an infinite margin", r0["f"] == 22.5 and r0["sd"] == 0.0 and r0["margin"] == math.inf))
    rm = L.recover_noisy_n([a30 + 0.2, a24 - 0.2, a25 + 0.1], [30, 24, 25], 0.3, 5, 399)
    checks.append((f"margin: a slightly noisy trial of 22.5 rev/s gives {rm['f']:.2f} and the best set beats the runner-up by {rm['margin']:.0f} sigma", abs(rm["f"] - 22.5) < 0.2 and rm["margin"] > 5))

    # ---- edges ----------------------------------------------------------------------------------------
    rng = np.random.default_rng(7)
    edge = {}
    for g in (5.0, 399.0):
        ok = 0
        for _ in range(500):
            a = [L.signed_alias(g, r) + rng.normal(0, 0.3) for r in (30, 24, 25)]
            r_ = L.recover_noisy_n(a, [30, 24, 25], 0.3, 5, 399)
            ok += abs(r_["f"] - g) <= 0.5 and 5 <= r_["f"] <= 399
        edge[g] = ok / 500
    checks.append((f"the edges of the range are recovered ({edge[5.0]:.0%} at 5 rev/s, {edge[399.0]:.0%} at 399 rev/s within 0.5) and the answer never leaves it", min(edge.values()) >= 0.97))
    errs = 0
    for fn in (lambda: L.common_speeds([a30], [30]),
               lambda: L.common_speeds([a30, a24], [30, 24, 25]),
               lambda: L.unambiguous_range([30, 24.5]),
               lambda: L.recover_noisy_n([a30, a24], [30, 24], -0.1),
               lambda: L.recover_noisy_n([a30, a24], [30, 24], 0.3, 399, 5),
               lambda: L.recover_noisy_n([a30, a24, a25], [30, 24, 25], 0.3, 0, 100000)):
        try:
            fn()
        except ValueError:
            errs += 1
    checks.append(("a single clip, mismatched lists, a fractional rate, a negative sigma, a reversed range and a search that is far too large are refused with a clear error", errs == 6))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
